# UNIDAD IV — Manejo de archivos, persistencia y validación

## CLASE 2: Archivos CSV – Manejo estructurado y formateado de datos

---

# PARTE I — FUNDAMENTOS TEÓRICOS (Antes de programar)

---



# 1. Del texto libre al dato estructurado

En la clase anterior guardamos:

```
Tarea 1
Tarea 2
Tarea 3
```

Problema:
No hay relación entre datos.

Ahora necesitamos guardar objetos:

| id | nombre  | precio | stock |
| -- | ------- | ------ | ----- |
| 1  | Mouse   | 40     | 10    |
| 2  | Teclado | 120    | 5     |

Esto ya no es texto libre.
Esto es:

> **Información tabular estructurada**

---

## ¿Qué es CSV?

CSV = **Comma Separated Values**

Es una representación de tablas en texto.

Ejemplo:

```
id,nombre,precio,stock
1,Mouse,40,10
2,Teclado,120,5
```

---

## Concepto importante

Un CSV es simultáneamente:

* legible por humanos
* legible por máquinas
* portable entre sistemas
* equivalente básico a una tabla de base de datos

Por eso es el formato intermedio más usado en ingeniería de datos.

---



# 2. Problemas reales del CSV (muy importante)

Un CSV no es tan simple como parece.

Ejemplo real incorrecto:

```
1,Monitor,24 pulgadas,800,5
```

¿Cuántas columnas hay?

→ Ambiguo

Solución:

```
1,"Monitor, 24 pulgadas",800,5
```

Aquí nace el módulo `csv` de Python:
no es comodidad → es seguridad de datos.

---


# 3. Separadores

No todos los CSV usan coma:

| País              | Separador |
| ----------------- | --------- |
| EEUU              | ,         |
| Europa            | ;         |
| Logs              | |         |
| Datos científicos | \t        |

Por eso:

> Nunca se debe parsear CSV con split(",")

---


# 4. Codificación (problema crítico en producción)

Ejemplo:

```
Café
Niño
Teléfono
```

Con encoding incorrecto:

```
Caf�
Ni�o
Tel�fono
```

Siempre usar:

```
encoding="utf-8"
newline=""
```

`newline=""` evita líneas vacías en Windows.

---


# 5. Lectura de CSV

## csv.reader

Convierte cada fila en lista

Archivo:

```
1,Mouse,40,10
```

Python:

```
['1','Mouse','40','10']
```

Problema:

No sabemos qué es cada dato.

---

## csv.DictReader

Convierte cada fila en diccionario

```
{'id': '1', 'nombre': 'Mouse', 'precio': '40', 'stock': '10'}
```

Esto es semánticamente correcto.

---


# 6. Escritura de CSV

## csv.writer

Escribe listas

## csv.DictWriter

Escribe diccionarios estructurados

Siempre preferir DictWriter en sistemas reales.

---




# 7. Problemas de calidad de datos

Archivos reales contienen:

| Problema          | Ejemplo            |
| ----------------- | ------------------ |
| Duplicados        | mismo ID           |
| Filas corruptas   | columnas faltantes |
| Tipos incorrectos | precio = "abc"     |
| Encoding roto     | caracteres raros   |
| Filas truncadas   | corte de energía   |

Por eso:

> leer CSV siempre debe validar

---




# 8. Validación automática de filas

Cada fila debe cumplir contrato:

```
id: entero positivo
nombre: texto no vacío
precio: float positivo
stock: entero >= 0
```

Si no → no se carga.

Esto se llama:

> saneamiento de datos (data sanitization)

---


# PARTE II — DISEÑO DE SOFTWARE


Ahora el archivo CSV será nuestra base de datos.

Arquitectura:

```
Presentación (menu)
Servicio (reglas inventario)
Repositorio (manejo CSV)
Infraestructura (csv manager)
Modelo (Producto)
Validación (sanitización)
```

---

## Patrón aplicado: Repository + Data Mapper

El CSV guarda texto
El sistema trabaja objetos

El repositorio traduce entre ambos mundos.

---




---

# PARTE III — IMPLEMENTACIÓN GUIADA



---

# 1. Modelo de dominio

## models/product.py


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Product:
    id: int
    name: str
    price: float
    stock: int


---

# 2. Validador

## utils/product_validator.py


In [ ]:
def validate_product(row: dict) -> dict:

    try:
        id_ = int(row["id"])
        if id_ <= 0:
            raise ValueError

        name = row["name"].strip()
        if not name:
            raise ValueError("Nombre vacío")

        price = float(row["price"])
        if price <= 0:
            raise ValueError

        stock = int(row["stock"])
        if stock < 0:
            raise ValueError

        return {
            "id": id_,
            "name": name,
            "price": price,
            "stock": stock
        }

    except Exception as e:
        raise ValueError(f"Fila inválida: {row}") from e


---

# 3. Infraestructura CSV segura

## utils/csv_manager.py


In [ ]:
import csv
from pathlib import Path
from typing import List, Dict

class CSVManager:

    def __init__(self, path: str, delimiter: str = ",") -> None:
        self._path = Path(path)
        self._delimiter = delimiter
        self._path.touch(exist_ok=True)

    def read_all(self) -> List[Dict]:

        with self._path.open("r", encoding="utf-8", newline="") as file:
            reader = csv.DictReader(file, delimiter=self._delimiter)
            return list(reader)

    def write_all(self, rows: List[Dict], headers: List[str]) -> None:

        with self._path.open("w", encoding="utf-8", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=headers, delimiter=self._delimiter)
            writer.writeheader()
            writer.writerows(rows)


---

# 4. Repository

## repository/product_repository.py


In [ ]:
from models.product import Product
from utils.csv_manager import CSVManager
from utils.product_validator import validate_product

class ProductRepository:

    HEADERS = ["id", "name", "price", "stock"]

    def __init__(self, path: str) -> None:
        self._csv = CSVManager(path)

    def get_all(self) -> list[Product]:
        rows = self._csv.read_all()
        products = []

        for row in rows:
            clean = validate_product(row)
            products.append(Product(**clean))

        return products

    def save_all(self, products: list[Product]) -> None:
        rows = [product.__dict__ for product in products]
        self._csv.write_all(rows, self.HEADERS)


---

# 5. Servicio de inventario

## services/inventory_service.py


In [ ]:
from repository.product_repository import ProductRepository
from models.product import Product

class InventoryService:

    def __init__(self, repo: ProductRepository) -> None:
        self._repo = repo

    def add_product(self, product: Product) -> None:
        products = self._repo.get_all()

        if any(p.id == product.id for p in products):
            raise ValueError("Producto duplicado")

        products.append(product)
        self._repo.save_all(products)

    def list_products(self) -> list[Product]:
        return self._repo.get_all()


---

# 6. Interfaz consola

## main.py


In [ ]:
from services.inventory_service import InventoryService
from repository.product_repository import ProductRepository
from models.product import Product

def main():

    service = InventoryService(ProductRepository("data/inventory.csv"))

    while True:
        print("\n1. Agregar producto")
        print("2. Listar productos")
        print("3. Salir")

        op = input("Seleccione: ")

        if op == "1":
            try:
                p = Product(
                    int(input("ID: ")),
                    input("Nombre: "),
                    float(input("Precio: ")),
                    int(input("Stock: "))
                )
                service.add_product(p)
                print("Guardado")
            except Exception as e:
                print("Error:", e)

        elif op == "2":
            for p in service.list_products():
                print(p)

        elif op == "3":
            break

if __name__ == "__main__":
    main()


---

# EJERCICIO PRÁCTICO

**Sistema de inventario persistente en CSV**

Debe permitir:

1. Cargar productos existentes
2. Agregar nuevos
3. Evitar duplicados
4. Validar filas corruptas
5. Mantener estructura del archivo
6. No perder datos al reescribir

---
